# ch01 数据清洗

本 Notebook 执行产品销售数据的清洗流程，包括缺失值处理、重复值删除、数据类型转换和数据验证。

## 1. 环境配置与数据加载

In [ ]:
import sys
from pathlib import Path

# 将项目根目录添加到 sys.path
PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.utils import load_raw_data, CATEGORY_LIST, CITY_LIST, OUTPUT_BASE

print('环境配置完成。')

In [ ]:
# 加载原始数据
df = load_raw_data()
print(f'\n原始数据形状: {df.shape}')
print(f'列名: {df.columns.tolist()}')
df.head()

## 2. 缺失值检查与处理

In [ ]:
# 检查缺失值
missing_summary = df.isnull().sum()
missing_total = missing_summary.sum()
print(f'缺失值总数: {missing_total}')
print('\n各列缺失值统计:')
print(missing_summary)

In [ ]:
# 处理缺失值：删除含有缺失值的行
if missing_total > 0:
    df = df.dropna()
    print(f'删除缺失值后数据形状: {df.shape}')
else:
    print('数据中无缺失值，无需处理。')

## 3. 重复值检查与处理

In [ ]:
# 检查重复值
duplicate_count = df.duplicated().sum()
print(f'重复行数: {duplicate_count}')

if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f'删除重复值后数据形状: {df.shape}')
else:
    print('数据中无重复行。')

## 4. 数据类型转换

In [ ]:
# 转换前数据类型
print('转换前数据类型:')
print(df.dtypes)

In [ ]:
# 转换日期列
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    print("已将 'date' 列转换为 datetime 类型。")

# 转换数值列
numeric_cols = ['price', 'quantity', 'total_price']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"已将 '{col}' 列转换为数值类型。")

# 转换类别列
if 'category' in df.columns:
    valid_categories = set(CATEGORY_LIST)
    df['category'] = df['category'].astype(str).str.strip()
    df = df[df['category'].isin(valid_categories)]
    df['category'] = df['category'].astype('category')
    print(f"已将 'category' 列转换为 category 类型。")

if 'city' in df.columns:
    valid_cities = set(CITY_LIST)
    df['city'] = df['city'].astype(str).str.strip()
    df = df[df['city'].isin(valid_cities)]
    df['city'] = df['city'].astype('category')
    print(f"已将 'city' 列转换为 category 类型。")

print(f'\n清洗后数据形状: {df.shape}')

In [ ]:
# 转换后数据类型
print('转换后数据类型:')
print(df.dtypes)

## 5. 保存清洗后的数据

In [ ]:
from src.utils import save_dataframe

output_dir = OUTPUT_BASE / 'data_cleaning'
cleaned_path = save_dataframe(df, output_dir / 'product_preprocessed.csv')
print(f'清洗后数据已保存至: {cleaned_path}')

## 6. 数据清洗总结

In [ ]:
print('=' * 50)
print('数据清洗总结')
print('=' * 50)
print(f'清洗后数据形状: {df.shape[0]} 行 x {df.shape[1]} 列')
print(f'数据列: {df.columns.tolist()}')
print(f'有效产品类别: {CATEGORY_LIST}')
print(f'有效城市: {CITY_LIST}')
print('=' * 50)
print('数据清洗完成，数据已保存至 outputs/data_cleaning/product_preprocessed.csv')